# SONA — Phase A remote batch (R2 timing · WER power fixes)

Three tasks that the local machine cannot do well:

| id | task | why not local |
|---|---|---|
| **R2** | ONNX timing: FP32 vs uniform INT8 vs stem-exempt mixed | local box runs ~20 % background load; absolute RTF is untrustworthy there. Kaggle is dedicated **and** Intel Xeon vs local AMD Ryzen, so this doubles as a cross-architecture replication |
| **W2** | WER on MS-SNSD (1100 clips) | needs a GPU Whisper pass |
| **W3** | WER on VoiceBank with `tiny.en` | needs a GPU Whisper pass |

**Why W2/W3 exist.** The per-clip WER tests came out underpowered: Whisper `small.en` transcribes
most of these short utterances perfectly under both methods, so only 129/824 pairs differ at all
and the per-condition effective sample falls to 16–37 clips. W2 adds clips; W3 raises the error
floor. Both attack the same problem — too few clips carrying signal.


In [ ]:
%pip install -q onnxruntime==1.20.1 faster-whisper jiwer pystoi

import glob, json, os, subprocess, sys, time
from pathlib import Path

IN_KAGGLE = Path('/kaggle/working').exists()
print('IN_KAGGLE =', IN_KAGGLE)
OUT_DIR = Path('/kaggle/working/phase_a' if IN_KAGGLE else './phase_a')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR =', OUT_DIR)

In [ ]:
def find_repo():
    here = Path.cwd()
    if (here / 'scripts' / 'bench_onnx_edge.py').exists():
        return str(here)
    hits = glob.glob('/kaggle/input/**/scripts/bench_onnx_edge.py', recursive=True)
    hits = sorted(hits, key=lambda p: ('sona' not in p.lower(), len(p)))
    return str(Path(hits[0]).parents[1]) if hits else None

REPO_DIR = find_repo()
print('REPO_DIR =', REPO_DIR)
assert REPO_DIR, 'upload the repo (sona.zip contents) as a Kaggle dataset'

def first(pattern, must=True):
    hits = sorted(glob.glob(pattern, recursive=True), key=len)
    if not hits and must:
        raise SystemExit('not found: ' + pattern)
    return hits[0] if hits else None

CHECKPOINT = first('/kaggle/input/**/ckpt_gtcrn_ch96_h96_b4_mw1p25_mr0p25/last.pt')
print('CHECKPOINT =', CHECKPOINT)

## Machine identity

Recorded so the timing numbers can be attributed. The paper must state CPU and load for any
absolute RTF it quotes — that was the fix applied after the first RTF pass proved load-inflated.


In [ ]:
import platform
info = {'platform': platform.platform(), 'processor': platform.processor(),
        'python': sys.version.split()[0]}
try:
    info['cpu_model'] = [l.split(':',1)[1].strip() for l in
                         open('/proc/cpuinfo') if l.startswith('model name')][0]
    info['cpu_count'] = sum(1 for l in open('/proc/cpuinfo') if l.startswith('processor'))
    info['loadavg_1min'] = float(open('/proc/loadavg').read().split()[0])
except Exception as e:
    info['cpuinfo_error'] = str(e)
print(json.dumps(info, indent=2))
(OUT_DIR / 'machine.json').write_text(json.dumps(info, indent=2))

## R2 — timing: FP32 vs uniform INT8 vs stem-exempt mixed

The claim under test: the mixed allocation carries **29 QDQ nodes against uniform INT8's 172**. If
QDQ graph overhead is what makes INT8 slower than FP32 here, mixed should sit between them — and
if it beats uniform, the stem exemption is smaller *and* faster, which turns a negative result
(INT8 is slower) into a positive one.


In [ ]:
ONNX_DIR = OUT_DIR / 'onnx_variants'
ONNX_DIR.mkdir(exist_ok=True)
import shutil
for name, pat in [('fp32', '/kaggle/input/**/model_fp32.onnx'),
                  ('int8', '/kaggle/input/**/model_int8.onnx'),
                  ('mixed', '/kaggle/input/**/frozen_mixed.onnx')]:
    src = first(pat)
    shutil.copy(src, ONNX_DIR / f'model_{name}.onnx')
    print(f'{name:6s} {os.path.getsize(src)/1024:9.1f} KB  <- {src}')

MANIFEST = first('/kaggle/input/**/test_manifest.csv')
print('MANIFEST =', MANIFEST)

In [ ]:
cmd = [sys.executable, str(Path(REPO_DIR) / 'scripts' / 'bench_onnx_edge.py'),
       '--checkpoint', CHECKPOINT,
       '--onnx-dir', str(ONNX_DIR),
       '--variants', 'fp32,int8,mixed',
       '--manifest', MANIFEST,
       '--limit', '100',
       '--threads', '1,2,4',
       '--out', str(OUT_DIR / 'r2_timing')]
print(' '.join(cmd), '\n')
subprocess.run(cmd, check=True, cwd=REPO_DIR)

In [ ]:
import pandas as pd
df = pd.read_csv(OUT_DIR / 'r2_timing' / 'summary.csv')
display(df.sort_values(['backend', 'threads']))

fp32 = df[(df.backend == 'fp32')].set_index('threads')['mean_rtf']
print('\nslowdown vs FP32 at matched thread count (>1 means slower):')
for b in ['int8', 'mixed']:
    sub = df[df.backend == b].set_index('threads')['mean_rtf']
    for t in sub.index:
        if t in fp32.index:
            print(f'  {b:6s} threads={t}  {sub[t]/fp32[t]:.3f}x')

## W2 — WER on MS-SNSD (1100 clips)

Doubles the ASR sample and adds noise categories VoiceBank does not have.


In [ ]:
MS_MANIFEST = first('/kaggle/input/**/ms_snsd_sona/test.csv')
print('MS_MANIFEST =', MS_MANIFEST)

# Manifests carry absolute Windows paths from the authoring machine; rewrite to Kaggle roots.
import csv, re
rows = list(csv.DictReader(open(MS_MANIFEST, newline='', encoding='utf-8')))
def relocate(p):
    name = Path(p.replace('\\', '/')).name
    hit = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True), key=len)
    return hit[0] if hit else None
fixed, missing = [], 0
for r in rows:
    c, n = relocate(r['clean']), relocate(r['noisy'])
    if c and n:
        r['clean'], r['noisy'] = c, n
        fixed.append(r)
    else:
        missing += 1
print(f'resolved {len(fixed)}/{len(rows)} rows ({missing} missing)')
MS_LOCAL = str(OUT_DIR / 'ms_snsd_test_kaggle.csv')
with open(MS_LOCAL, 'w', newline='', encoding='utf-8') as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(fixed)
print('wrote', MS_LOCAL)

In [ ]:
cmd = [sys.executable, str(Path(REPO_DIR) / 'scripts' / 'eval_wer.py'),
       '--manifest', MS_LOCAL,
       '--limit', '0',
       '--whisper-model', 'small.en',
       '--gtcrn-checkpoint', CHECKPOINT,
       '--out', str(OUT_DIR / 'w2_wer_ms_snsd')]
print(' '.join(cmd), '\n')
subprocess.run(cmd, check=True, cwd=REPO_DIR)
print((OUT_DIR / 'w2_wer_ms_snsd' / 'summary_wer.md').read_text())

## W3 — WER on VoiceBank with a weaker ASR model

`small.en` is near-ceiling on this corpus, which is *why* the per-condition tests were
underpowered. `tiny.en` raises the error floor so more clips carry signal.

Reporting both is the honest framing: whether enhancement helps ASR is genuinely
ASR-model-dependent, and that dependence is itself on-thesis for a paper about metric choice
changing conclusions.


In [ ]:
cmd = [sys.executable, str(Path(REPO_DIR) / 'scripts' / 'eval_wer.py'),
       '--manifest', MANIFEST,
       '--limit', '0',
       '--whisper-model', 'tiny.en',
       '--gtcrn-checkpoint', CHECKPOINT,
       '--out', str(OUT_DIR / 'w3_wer_tiny')]
print(' '.join(cmd), '\n')
subprocess.run(cmd, check=True, cwd=REPO_DIR)
print((OUT_DIR / 'w3_wer_tiny' / 'summary_wer.md').read_text())

In [ ]:
# effective-sample check: the whole point of W3
import pandas as pd
d = pd.read_csv(OUT_DIR / 'w3_wer_tiny' / 'per_sample_wer.csv')
piv = d.pivot_table(index='uid', columns='method', values='wer')
if {'neural_gtcrn', 'passthrough'} <= set(piv.columns):
    diff = (piv['neural_gtcrn'] - piv['passthrough']).dropna()
    print(f'clips: {len(diff)}   differing: {(diff != 0).sum()}   '
          f'({100*(diff != 0).mean():.1f} %)')
    print('small.en reference was 129/824 = 15.7 %')

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/phase_a_results' if IN_KAGGLE else './phase_a_results',
                    'zip', OUT_DIR)
print('bundled ->', 'phase_a_results.zip')